# MaxPreps High School Football Statistics Scraper

This notebook harvests high school football player statistics from MaxPreps using Selenium with anti-blocking strategies.

## Features
- **Multi-browser support**: Automatically selects Edge (Windows) or Chrome (Linux/GCP)
- **Years supported**: 2015-2025
- **Categories**: Passing, Rushing, Receiving, Tackles, Interceptions, Returns
- **Anti-blocking**: User-agent rotation, rate limiting, automation detection disabled
- **Pagination**: Scrapes ~500 records per category (10 pages × 50 records)
- **Progress tracking**: Resumes from last checkpoint if interrupted

In [10]:
# Add parent directory to Python path so we can import from src
import sys
import os

# Get the notebook directory and add parent (project root) to path
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))

# Add to path if not already there
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"Python path updated: {sys.path[0]}")

Project root: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence
Python path updated: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence


In [11]:
import importlib
from scraper_utils import maxpreps_scraper

# Force reload of the module to ensure latest changes are loaded
importlib.reload(maxpreps_scraper)

from scraper_utils.maxpreps_scraper import MaxPrepsScraper

print("✓ MaxPrepsScraper imported successfully (module reloaded)")
print(f"  Edge driver paths: {maxpreps_scraper.SeleniumDriverManager.EDGE_PATHS}")

✓ MaxPrepsScraper imported successfully (module reloaded)
  Edge driver paths: ['X:\\Python\\edge_driver\\msedgedriver.exe', 'C:\\Python\\edge_driver\\msedgedriver.exe']


## Configuration

Configure what you want to scrape:

In [12]:
# ============================================================================
# CONFIGURATION - Customize scraping parameters here
# ============================================================================

# Year range to scrape (examples: [2025], [2023, 2024, 2025], [2015-2026])
YEARS = list(range(2011, 2026))  # All years 2015-2025 (11 years)

# Statistics categories to scrape
CATEGORIES = ['passing', 'rushing', 'receiving', 'tackles', 'interceptions', 'returns','sacks']
# Example: CATEGORIES = ['passing']  # Single category

# Pages per category (each page has ~50 records)
# Default: 10 pages = ~500 records per category/year
PAGES_PER_CATEGORY = 10

# Overwrite mode - how to handle existing files:
#   "all"        - Delete all files and re-scrape everything
#   "missing"    - Only scrape files that don't exist yet
#   "incomplete" - Scrape files that don't exist OR have fewer than target records (DEFAULT)
#                  For incomplete files, resumes from where they left off
OVERWRITE_MODE = "incomplete"

# Display settings
HEADLESS = True  # Set to False to see the browser (for debugging)

# Use absolute path for data output (based on project root)
DATA_OUTPUT_DIR = os.path.join(project_root, "data", "high_school_stats")

# Calculate target records
TARGET_RECORDS_PER_CATEGORY = PAGES_PER_CATEGORY * 50

print("╔" + "="*68 + "╗")
print("║" + " "*68 + "║")
print("║" + "MAXPREPS SCRAPER CONFIGURATION".center(68) + "║")
print("║" + " "*68 + "║")
print("╚" + "="*68 + "╝")
print(f"\nYear Range:")
print(f"  {YEARS[0]} to {YEARS[-1]} ({len(YEARS)} years)")

print(f"\nCategories:")
for i, cat in enumerate(CATEGORIES, 1):
    print(f"  {i}. {cat}")

print(f"\nPages & Records:")
print(f"  Pages per category: {PAGES_PER_CATEGORY}")
print(f"  Target records per category/year: {TARGET_RECORDS_PER_CATEGORY}")
print(f"  Total combinations: {len(YEARS)} × {len(CATEGORIES)} = {len(YEARS) * len(CATEGORIES)}")
print(f"  Estimated total records (if all complete): {len(YEARS) * len(CATEGORIES) * TARGET_RECORDS_PER_CATEGORY:,}")

print(f"\nOverwrite Mode: {OVERWRITE_MODE}")
if OVERWRITE_MODE == "all":
    print(f"  → Will DELETE and re-scrape ALL files")
elif OVERWRITE_MODE == "missing":
    print(f"  → Will ONLY scrape files that don't exist yet")
elif OVERWRITE_MODE == "incomplete":
    print(f"  → Will scrape files with < {TARGET_RECORDS_PER_CATEGORY} records")
    print(f"  → For incomplete files, will resume from where they stopped")

print(f"\nOutput Directory: {DATA_OUTPUT_DIR}")
print("\n" + "="*70)

╔====================================================================╗
║                                                                    ║
║                   MAXPREPS SCRAPER CONFIGURATION                   ║
║                                                                    ║
╚====================================================================╝

Year Range:
  2011 to 2025 (15 years)

Categories:
  1. passing
  2. rushing
  3. receiving
  4. tackles
  5. interceptions
  6. returns
  7. sacks

Pages & Records:
  Pages per category: 10
  Target records per category/year: 500
  Total combinations: 15 × 7 = 105
  Estimated total records (if all complete): 52,500

Overwrite Mode: incomplete
  → Will scrape files with < 500 records
  → For incomplete files, will resume from where they stopped

Output Directory: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\high_school_stats



In [13]:
# Check existing files and what will be scraped
from pathlib import Path
import pandas as pd

print("\n" + "="*70)
print("EXISTING FILES STATUS")
print("="*70)

data_dir = Path(DATA_OUTPUT_DIR)
file_status = {}
temp_status = {}

# Check status of all year/category combinations
for category in CATEGORIES:
    category_path = data_dir / category
    if not category_path.exists():
        continue
    
    if category not in file_status:
        file_status[category] = {}
    if category not in temp_status:
        temp_status[category] = {}
    
    for year in YEARS:
        csv_file = category_path / f"{year}.csv"
        temp_file = category_path / f"{year}.tmp.csv"
        
        # Check temp file first
        if temp_file.exists():
            try:
                temp_row_count = len(pd.read_csv(temp_file))
                resume_page = (temp_row_count // 50) + 1
                temp_status[category][year] = (temp_row_count, resume_page)
                print(f"⏸  {category:15s} {year}: TEMP FILE with {temp_row_count} records (resume page {resume_page})")
            except Exception as e:
                print(f"✗ {category:15s} {year}: Error reading temp file - {e}")
        elif csv_file.exists():
            try:
                row_count = len(pd.read_csv(csv_file))
                status = "COMPLETE" if row_count >= TARGET_RECORDS_PER_CATEGORY else "INCOMPLETE"
                status_color = "✓" if status == "COMPLETE" else "⚠"
                file_status[category][year] = (row_count, status)
                print(f"{status_color} {category:15s} {year}: {row_count:3d}/{TARGET_RECORDS_PER_CATEGORY:3d} records [{status}]")
            except Exception as e:
                print(f"✗ {category:15s} {year}: Error reading file - {e}")
        else:
            file_status[category][year] = (0, "MISSING")
            print(f"○ {category:15s} {year}: No file [MISSING]")

print("\n" + "="*70)
print("SCRAPING PLAN")
print("="*70)
print(f"Overwrite mode: {OVERWRITE_MODE}")

if OVERWRITE_MODE == "incomplete":
    to_scrape = []
    to_resume = []
    to_resume_temp = []
    
    for category in CATEGORIES:
        for year in YEARS:
            # Check if there's a temp file to resume
            if category in temp_status and year in temp_status[category]:
                temp_rows, resume_page = temp_status[category][year]
                to_resume_temp.append((year, category, temp_rows, resume_page))
            elif category not in file_status or year not in file_status[category]:
                to_scrape.append((year, category, "NEW"))
            else:
                row_count, status = file_status[category][year]
                if status == "MISSING":
                    to_scrape.append((year, category, "NEW"))
                elif status == "INCOMPLETE":
                    to_resume.append((year, category, row_count))
    
    print(f"\nNew files to scrape: {len(to_scrape)}")
    print(f"Incomplete files to resume: {len(to_resume)}")
    print(f"Temp files to resume: {len(to_resume_temp)}")
    
    if to_resume_temp:
        print("\n⏸ TEMP FILES (partial scrapes in progress):")
        for year, category, temp_rows, resume_page in to_resume_temp:
            print(f"  • {category:15s} {year}: {temp_rows} records → resume page {resume_page}")
    
    if to_resume:
        print("\nIncomplete files to resume:")
        for year, category, existing_count in to_resume:
            start_page = (existing_count // 50) + 1
            print(f"  • {category:15s} {year}: {existing_count} records → resume page {start_page}")

elif OVERWRITE_MODE == "missing":
    to_scrape = []
    to_resume_temp = []
    
    for category in CATEGORIES:
        for year in YEARS:
            if category in temp_status and year in temp_status[category]:
                temp_rows, resume_page = temp_status[category][year]
                to_resume_temp.append((year, category, temp_rows, resume_page))
            elif category not in file_status or year not in file_status[category] or file_status[category][year][1] == "MISSING":
                to_scrape.append((year, category))
    
    print(f"\nFiles to scrape (non-existent only): {len(to_scrape)}")
    if to_resume_temp:
        print(f"Temp files to resume: {len(to_resume_temp)}")
        for year, category, temp_rows, resume_page in to_resume_temp:
            print(f"  • {category:15s} {year}: {temp_rows} records → resume page {resume_page}")

elif OVERWRITE_MODE == "all":
    print(f"\nWill DELETE and RE-SCRAPE all {len(YEARS) * len(CATEGORIES)} combinations")

print("\n" + "="*70 + "\n")


EXISTING FILES STATUS

SCRAPING PLAN
Overwrite mode: incomplete

New files to scrape: 105
Incomplete files to resume: 0
Temp files to resume: 0




## Quick Start Guide

### Overwrite Modes Explained

Choose the mode that matches your needs in the configuration cell above:

**`overwrite="incomplete"` (DEFAULT - Recommended)**
- Scrapes files that don't exist
- Scrapes files with fewer than target records
- **Resumes from where incomplete files left off** (no re-scraping!)
- Example: 2025 passing has 450 records → only scrapes page 10 to reach 500 records

**`overwrite="missing"`**
- Only scrapes files that don't exist yet
- Skips all existing files (even if incomplete)
- Use this if you want to preserve partial data while scraping new years/categories

**`overwrite="all"`**
- Deletes all existing files first, then re-scrapes everything
- Use this if you want a completely fresh scrape

## Refactored Scraper - Key Improvements

🔧 **Major fixes applied to resolve 100% duplicate data issue:**

1. **Fixed HTML Player Metadata Extraction** ✓
   - Now correctly parses complex NAME column HTML structure
   - Extracts: Player name, Position, School, and State (from school HTML)
   - Handles malformed HTML with robust fallbacks

2. **Button-Based Pagination Fix** ✓
   - Removed unreliable URL parameter approach
   - Now uses robust Selenium button clicking for page navigation
   - Properly advances through unique player data on each page

3. **Per-Page Duplicate Detection** ✓
   - Real-time monitoring during scraping
   - Detects if pagination fails (80%+ row overlap = same page repeated)
   - Stops immediately instead of collecting thousands of duplicate rows

4. **Enhanced Data Enrichment** ✓
   - Added `State` column (extracted from school field)
   - Added `Year` column (from URL/configuration)
   - Added `Category` column (stat type: passing, rushing, etc.)

5. **New File Naming** ✓
   - Old: `{category}/{year}.csv`
   - New: `hs_{category}_{year}.csv` (flat, descriptive)
   - Example: `hs_passing_2025.csv`

**Test Results:**
- ✓ Successfully extracted 100 unique rows (2 pages)
- ✓ Page 1: Ranks 1-50, Page 2: Ranks 51-100 (no duplicates!)
- ✓ All metadata columns populated correctly

In [14]:
# Test: Scrape 2025 passing stats (10 pages = ~500 records) in headless mode
print("Testing refactored scraper with 2025 passing stats (10 pages = ~500 records)...\n")

test_scraper = MaxPrepsScraper(headless=False, output_dir=DATA_OUTPUT_DIR)
test_scraper.run(
    years=[2025],
    categories=['passing'],
    pages_per_category=10,
    overwrite="all"  # Fresh scrape for testing
)

print(f"\n✓ Test complete! Check {DATA_OUTPUT_DIR}/hs_passing_2025.csv")

INFO: Starting MaxPreps scraper
INFO:   Years: 2025-2025 (1 years)
INFO:   Categories: ['passing']
INFO:   Pages per category: 10
INFO:   Target records: 500 per category/year
INFO:   Overwrite mode: all
INFO: Scraping 2025/passing (pages 1-10)
INFO: Using Chrome driver for MaxPreps scraping


Testing refactored scraper with 2025 passing stats (10 pages = ~500 records)...



INFO: Chrome driver initialized with user-agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWeb... (eager page load, ad blocking enabled)
INFO:   Page 1/10
--- Logging error ---
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3312.0_x64__qbz5n2kfra8p0\Lib\logging\__init__.py", line 1154, in emit
    stream.write(msg + self.terminator)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3312.0_x64__qbz5n2kfra8p0\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2713' in position 48: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "x:\Python\.venv\Lib\site-pa


✓ Test complete! Check x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\high_school_stats/hs_passing_2025.csv


## Full Scrape (All Years & Categories)

Run this to scrape all data. This will take many hours.

**To interrupt**: Press `Ctrl+C` in the notebook. The scraper saves progress after each page, so you can resume later by running the same code again.

In [ ]:
# Full scrape using configuration from above
import time
from datetime import datetime

print("\n" + "="*70)
print("STARTING SCRAPER")
print("="*70)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Overwrite mode: {OVERWRITE_MODE}")
print("To interrupt: Press Ctrl+C in the notebook")
print("Progress is saved after each page, so you can resume later.\n")

scraper = MaxPrepsScraper(headless=HEADLESS, output_dir=DATA_OUTPUT_DIR)

try:
    scraper.run(
        years=YEARS,
        categories=CATEGORIES,
        pages_per_category=PAGES_PER_CATEGORY,
        overwrite=OVERWRITE_MODE
    )
    print("\n" + "="*70)
    print("✓ SCRAPING COMPLETE!")
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*70)
except KeyboardInterrupt:
    print("\n\n⚠ Scraping interrupted by user")
    print(f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("You can resume by running this cell again - it will pick up where it left off.")

## Verify Output

Check what was scraped:

In [ ]:
import pandas as pd
from pathlib import Path

# Find all CSV files (new flat structure)
data_dir = Path(DATA_OUTPUT_DIR)
csv_files = list(data_dir.glob("hs_*.csv"))

print(f"Found {len(csv_files)} CSV files in: {data_dir}\n")

# Summary by category and year
summary = {}
for csv_file in sorted(csv_files):
    # Parse filename: hs_{category}_{year}.csv
    filename = csv_file.stem  # Remove .csv
    parts = filename.split('_')
    if len(parts) >= 3:
        category = parts[1]
        year = parts[2]
        
        # Read and count rows
        df = pd.read_csv(csv_file)
        row_count = len(df)
        
        if category not in summary:
            summary[category] = {}
        summary[category][year] = row_count
        
        print(f"{category:15s} {year} -> {row_count:4d} records - {csv_file.name}")

if summary:
    print(f"\n{'='*60}")
    print("Summary by category:")
    for category in sorted(summary.keys()):
        total = sum(summary[category].values())
        years = len(summary[category])
        avg_per_year = total // years if years > 0 else 0
        print(f"  {category:15s}: {total:6,d} records across {years:2d} years ({avg_per_year:3d} avg/year)")
    
    grand_total = sum(sum(v.values()) for v in summary.values())
    total_years = len(set(year for cat in summary.values() for year in cat.keys()))
    print(f"\nGrand total: {grand_total:,} records across {total_years} years")
else:
    print("No CSV files found. Run the test or full scrape first.")

## Sample Data

View sample from one of the scraped files:

In [ ]:
# Load and display sample data
sample_file = Path(DATA_OUTPUT_DIR) / "hs_passing_2025.csv"
if sample_file.exists():
    df = pd.read_csv(sample_file)
    print(f"Sample from {sample_file}:")
    print(f"Shape: {df.shape}")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nFirst 5 rows:")
    display(df.head())
else:
    print(f"File not found: {sample_file}")
    print("Run the test cell above first.")

## Output Directory Structure

Files are saved in a flat structure with descriptive naming:

```
data/high_school_stats/
├── hs_passing_2015.csv
├── hs_passing_2016.csv
├── ...
├── hs_passing_2025.csv
├── hs_rushing_2015.csv
├── hs_rushing_2016.csv
├── ...
├── hs_receiving_2015.csv
├── hs_tackles_2015.csv
├── hs_interceptions_2015.csv
└── hs_returns_2015.csv
```

## CSV Schema

Each CSV contains:
- **Rank**: Player's rank in that stat
- **Player**: Player name
- **Position**: Position(s)
- **School**: High school name
- **State**: Player state (extracted from school data)
- **Year**: Season year
- **Category**: Stat category (passing, rushing, etc.)
- **Stats**: Dynamic columns based on category (e.g., Passing has YDS, COMP, ATT, PCT, TD, INT, RATE, GP, etc.)

## Progress Tracking

A `scraper_progress.json` file tracks completion status. If interrupted, run the scraper again and it will skip already-completed categories and resume from where it left off.

## Data Quality Check: Duplicate Row Detection

Check for duplicate rows in raw high school stats files. Many files show the same 50 players repeated, indicating incomplete scrapes that need to be re-run.

In [ ]:
import os
import pandas as pd
from pathlib import Path

# Check raw files for duplicates
raw_stats_dir = os.path.join(project_root, "data", "high_school_stats_raw")

if not os.path.exists(raw_stats_dir):
    raw_stats_dir = "data/high_school_stats_raw"

print(f"Checking for duplicate rows in: {os.path.abspath(raw_stats_dir)}\n")
print("="*80)

duplicate_report = []
rescrape_list = []

# Categories to check
categories = ['passing', 'receiving', 'rushing', 'tackles', 'interceptions', 'returns', 'sacks']

for category in categories:
    category_dir = os.path.join(raw_stats_dir, category)
    
    if not os.path.exists(category_dir):
        print(f"✗ Category directory not found: {category}")
        continue
    
    print(f"\n{category.upper()}")
    print("-" * 80)
    
    csv_files = sorted(Path(category_dir).glob("*.csv"))
    
    for filepath in csv_files:
        try:
            df = pd.read_csv(filepath)
            year = filepath.stem
            
            # Count total rows and duplicate rows
            total_rows = len(df)
            duplicate_rows = total_rows - len(df.drop_duplicates())
            unique_rows = len(df.drop_duplicates())
            
            # Check if there are duplicates
            if duplicate_rows > 0:
                dup_pct = (duplicate_rows / total_rows) * 100
                status = f"⚠ {duplicate_rows} duplicates ({dup_pct:.1f}%)"
                rescrape_list.append(f"{category}/{year}")
                color = "⚠"
            else:
                status = "✓ OK"
                color = "✓"
            
            duplicate_report.append({
                'Category': category,
                'Year': year,
                'Total Rows': total_rows,
                'Unique Rows': unique_rows,
                'Duplicate Rows': duplicate_rows,
                'Needs Rescrape': 'YES' if duplicate_rows > 0 else 'NO'
            })
            
            print(f"  {color} {year}: {total_rows} total, {unique_rows} unique, {duplicate_rows} duplicates - {status}")
        
        except Exception as e:
            print(f"  ✗ Error reading {filepath.name}: {e}")

# Create summary dataframe
summary_df = pd.DataFrame(duplicate_report)

print("\n" + "="*80)
print("DUPLICATE SUMMARY")
print("="*80)
print(summary_df.to_string(index=False))

# Files that need rescraped
print("\n" + "="*80)
print("FILES REQUIRING RESCRAPE")
print("="*80)

if rescrape_list:
    print(f"\nTotal files with duplicates: {len(rescrape_list)}\n")
    for file_path in rescrape_list:
        print(f"  • {file_path}")
else:
    print("\n✓ No files with duplicates found!")

print("\n" + "="*80)

# Summary stats
total_files = len(summary_df)
files_with_dups = len(rescrape_list)
total_dup_rows = summary_df['Duplicate Rows'].sum()

print(f"\nOVERALL STATISTICS:")
print(f"  Total files checked: {total_files}")
print(f"  Files with duplicates: {files_with_dups}")
print(f"  Files OK: {total_files - files_with_dups}")
print(f"  Total duplicate rows: {total_dup_rows}")
print("="*80)

In [26]:

import requests
import re
import time
import random
import json

class MaxPrepsAPIScraper:
    """Direct JSON API scraper for MaxPreps - bypasses Selenium pagination issues."""
    
    def __init__(self):
        self.session = requests.Session()
        # Mimic a real browser to avoid 403 Forbidden errors
        self.session.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Accept": "*/*",
            "Referer": "https://www.maxpreps.com/",
            "x-nextjs-data": "1"  # Tells server we want JSON, not HTML
        })
        self.build_id = self._get_build_id()
        
    def _get_build_id(self):
        """
        Dynamically fetch the current Build ID from the MaxPreps homepage.
        Extracts only digits to ensure clean numeric ID.
        """
        print("Fetching current Build ID...")
        try:
            r = requests.get("https://www.maxpreps.com/", 
                           headers={"User-Agent": self.session.headers["User-Agent"]},
                           timeout=10)
            # Look for the JSON blob Next.js embeds in the HTML
            match = re.search(r'"buildId":"(.*?)"', r.text)
            if match:
                # Extract only digits (removes \n, spaces, and other junk)
                build_id = ''.join(c for c in match.group(1) if c.isdigit())
                print(f"  Success! Build ID: {build_id}")
                return build_id
            else:
                print("  WARNING: Could not find buildId in page source, using fallback")
                return "1770310387"  # Fallback
        except Exception as e:
            print(f"  Error fetching Build ID: {e}, using fallback")
            return "1770310387"

    def _find_player_rows(self, data_dict, debug=False):
        """
        Recursively search for player rows in the JSON response.
        MaxPreps API nests player data deeply - this handles multiple possible structures.
        """
        # Pattern 1: leaders.rows (most common)
        if isinstance(data_dict, dict):
            if 'leaders' in data_dict and isinstance(data_dict['leaders'], dict):
                if 'rows' in data_dict['leaders']:
                    return data_dict['leaders']['rows']
            
            # Pattern 2: data.rows
            if 'data' in data_dict and isinstance(data_dict['data'], dict):
                if 'rows' in data_dict['data']:
                    return data_dict['data']['rows']
            
            # Pattern 3: Direct rows array
            if 'rows' in data_dict and isinstance(data_dict['rows'], list):
                if len(data_dict['rows']) > 0:
                    return data_dict['rows']
            
            # Pattern 4: Recursively search all dict values
            for key, value in data_dict.items():
                if isinstance(value, dict):
                    result = self._find_player_rows(value, debug=False)
                    if result is not None:
                        return result
        
        return None

    def scrape_category(self, year_label, category_path, stat_name, debug=False):
        """
        Scrape a specific category via the actual stat-leaders page.
        Extracts embedded __NEXT_DATA__ and retrieves statLeadersListData (all 500 players).
        
        Args:
            year_label: '24-25', '23-24', or 'current' (for URL construction)
            category_path: 'offense/passing/yds' (URL path)
            stat_name: 'passing' (for filename)
            debug: If True, print JSON structure for debugging
        """
        all_data = []
        
        # Construct the actual page URL
        # 2025 (Current) -> /football/stat-leaders/offense/passing/yds/
        # Previous     -> /football/24-25/stat-leaders/offense/passing/yds/
        if year_label == "current":
            url = f"https://www.maxpreps.com/football/stat-leaders/{category_path}/"
        else:
            url = f"https://www.maxpreps.com/football/{year_label}/stat-leaders/{category_path}/"
            
        print(f"\n--- Scraping {stat_name} ({year_label}) ---")
        print(f"  URL: {url}")
        
        try:
            print(f"  Fetching...", end=" ", flush=True)
            r = self.session.get(url, timeout=15)
            
            if r.status_code != 200:
                print(f"Failed ({r.status_code})")
                return all_data
            
            # Extract embedded Next.js JSON from HTML
            match = re.search(r'<script id="__NEXT_DATA__" type="application/json">(.*?)</script>', r.text, re.DOTALL)
            if not match:
                print("Could not find embedded __NEXT_DATA__ in HTML")
                return all_data
            
            data = json.loads(match.group(1))
            props = data.get('props', {}).get('pageProps', {})
            
            # Look for statLeadersListData (contains all 500 players at once)
            if 'statLeadersListData' in props:
                all_data = props['statLeadersListData']
                print(f"Found {len(all_data)} players in statLeadersListData")
            else:
                print("statLeadersListData not found in pageProps")
                print(f"  Available keys: {list(props.keys())[:10]}")
            
        except requests.exceptions.Timeout:
            print("Timeout")
        except Exception as e:
            print(f"Error: {e}")
        
        # Save to CSV
        if all_data:
            df = pd.DataFrame(all_data)
            
            # Determine output year from label
            if year_label == "current":
                output_year = 2025
            else:
                output_year = 2000 + int(year_label.split('-')[0])
            
            # Use consistent output directory
            output_file = os.path.join(DATA_OUTPUT_DIR, f"api_hs_{stat_name}_{output_year}.csv")
            df.to_csv(output_file, index=False)
            print(f"  Saved {len(df)} rows to {output_file}")
            print(f"  Columns: {list(df.columns)}")
            return len(all_data)
        else:
            print("  No data scraped")
            return 0

# Test the API scraper
print("="*70)
print("TESTING MAXPREPS JSON API SCRAPER (Updated)")
print("="*70)

api_scraper = MaxPrepsAPIScraper()
print(f"\nAPI Build ID successfully fetched: {api_scraper.build_id}\n")

# Test with 2025 passing stats (current season)
# MaxPreps API paths: offense/passing/yds, offense/rushing/yds, etc.
print("Testing API with 2025 passing stats (fetches all 500 at once)...")
rows_scraped = api_scraper.scrape_category(
    "current", 
    "offense/passing/yds", 
    "passing",
    debug=True  # Show available keys if data structure is different
)

print(f"\n{'='*70}")
print(f"API Test Complete: Scraped {rows_scraped} rows")
print(f"Output dir: {DATA_OUTPUT_DIR}")
if rows_scraped > 0:
    print("SUCCESS: API scraper is working!")
else:
    print("WARNING: No rows scraped - check the JSON structure above")
print(f"{'='*70}")


TESTING MAXPREPS JSON API SCRAPER (Updated)
Fetching current Build ID...
  Success! Build ID: 1770310387

API Build ID successfully fetched: 1770310387

Testing API with 2025 passing stats (fetches all 500 at once)...

--- Scraping passing (current) ---
  URL: https://www.maxpreps.com/football/stat-leaders/offense/passing/yds/
  Fetching... Found 14 players in statLeadersListData


ValueError: All arrays must be of the same length

## API Debugging: Discovering the Actual Endpoint

The initial API test returned 404, which means the `_next/data/` endpoint doesn't work for stat pages. Let's diagnose the actual API structure MaxPreps uses.

In [20]:
import requests
import re

# Test different possible API endpoints
print("="*70)
print("DIAGNOSTIC: Testing MaxPreps API Endpoints")
print("="*70)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "x-nextjs-data": "1"
}

# Get build ID first
r = requests.get("https://www.maxpreps.com/", headers=headers, timeout=10)
build_id_match = re.search(r'"buildId":"(.*?)"', r.text)
# Extract only digits (removes \n, spaces, and other junk)
build_id = ''.join(c for c in build_id_match.group(1) if c.isdigit()) if build_id_match else "unknown"
print(f"\nBuild ID: {build_id}")

print("\n--- URL Patterns to Test ---\n")

# Test 1: Direct _next/data endpoint (what we tried)
urls_to_test = [
    # Pattern 1: _next/data/buildId/ structure
    (f"https://www.maxpreps.com/_next/data/{build_id}/football/stat-leaders/offense/passing/yds.json", 
     "_next/data with full path"),
    
    # Pattern 2: Without stat type (just offense/passing)
    (f"https://www.maxpreps.com/_next/data/{build_id}/football/stat-leaders/offense/passing.json", 
     "_next/data without yds"),
    
    # Pattern 3: Alternative API endpoint
    ("https://www.maxpreps.com/api/stat-leaders?sport=football&stat=passing", 
     "Direct /api/stat-leaders endpoint"),
    
    # Pattern 4: GraphQL endpoint
    ("https://www.maxpreps.com/graphql", 
     "GraphQL endpoint"),
    
    # Pattern 5: Legacy stats API
    ("https://www.maxpreps.com/api/v2/stat-leaders/passing?year=2025", 
     "API v2 endpoint"),
]

for url, description in urls_to_test:
    try:
        print(f"Testing: {description}")
        print(f"  URL: {url}")
        r = requests.get(url, headers=headers, timeout=5)
        print(f"  Status: {r.status_code}")
        
        # If it's JSON, show structure
        if r.status_code == 200 and 'json' in r.headers.get('content-type', ''):
            try:
                data = r.json()
                if isinstance(data, dict):
                    print(f"  Keys: {list(data.keys())[:5]}...")  # Show first 5 keys
                    if 'pageProps' in data:
                        print(f"  pageProps keys: {list(data['pageProps'].keys())[:5]}...")
                elif isinstance(data, list):
                    print(f"  Data is list with {len(data)} items")
            except:
                print(f"  Response length: {len(r.text)} chars")
        else:
            print(f"  Response length: {len(r.text)} chars")
        
        print()
    except Exception as e:
        print(f"  Error: {e}\n")

print("\n--- Analyzing Selenium Approach ---")
print("Since the API endpoints aren't giving us direct access,")
print("the Selenium scraper working on pages 1-3 suggests:")
print("  1. Data IS on the page (loaded client-side or in HTML)")
print("  2. But pagination (arrow buttons) fails on page 4+")
print("\nPossible fixes for Selenium approach:")
print("  • Use execute_script to scroll pagination container")
print("  • Wait longer for JS-rendered pagination")
print("  • Check if arrow button becomes disabled on page 4")


DIAGNOSTIC: Testing MaxPreps API Endpoints

Build ID: 1770310387

--- URL Patterns to Test ---

Testing: _next/data with full path
  URL: https://www.maxpreps.com/_next/data/1770310387/football/stat-leaders/offense/passing/yds.json
  Status: 200
  Keys: ['pageProps', '__N_SSP']...
  pageProps keys: ['canonicalUrl', 'tracking', 'requestContext', 'pageTitle', 'pageDescription']...

Testing: _next/data without yds
  URL: https://www.maxpreps.com/_next/data/1770310387/football/stat-leaders/offense/passing.json
  Status: 200
  Keys: ['pageProps', '__N_SSP']...
  pageProps keys: ['canonicalUrl', 'tracking', 'requestContext', 'pageTitle', 'pageDescription']...

Testing: Direct /api/stat-leaders endpoint
  URL: https://www.maxpreps.com/api/stat-leaders?sport=football&stat=passing
  Status: 404
  Response length: 44328 chars

Testing: GraphQL endpoint
  URL: https://www.maxpreps.com/graphql
  Status: 404
  Response length: 93547 chars

Testing: API v2 endpoint
  URL: https://www.maxpreps.com/ap

In [27]:
base_url = "https://www.maxpreps.com/_next/data/1770310387/football"
endpoint = "stat-leaders/offense/passing.json"

# Loop from 2015 to 2025 (The current season is 2025)
for year in range(2015, 2026):
    # Format the season string (e.g., 2024 -> "24-25")
    short_start = str(year)[-2:]
    short_end = str(year + 1)[-2:]
    season_code = f"{short_start}-{short_end}"
    
    # URL Logic
    if year == 2025:
        # 2025 is the "Current" season, so it typically has no year code in the path
        full_url = f"{base_url}/{endpoint}"
    else:
        # Past years need the year code inserted
        full_url = f"{base_url}/{season_code}/{endpoint}"
        
    print(f"Season {year}: {full_url}")

Season 2015: https://www.maxpreps.com/_next/data/1770310387/football/15-16/stat-leaders/offense/passing.json
Season 2016: https://www.maxpreps.com/_next/data/1770310387/football/16-17/stat-leaders/offense/passing.json
Season 2017: https://www.maxpreps.com/_next/data/1770310387/football/17-18/stat-leaders/offense/passing.json
Season 2018: https://www.maxpreps.com/_next/data/1770310387/football/18-19/stat-leaders/offense/passing.json
Season 2019: https://www.maxpreps.com/_next/data/1770310387/football/19-20/stat-leaders/offense/passing.json
Season 2020: https://www.maxpreps.com/_next/data/1770310387/football/20-21/stat-leaders/offense/passing.json
Season 2021: https://www.maxpreps.com/_next/data/1770310387/football/21-22/stat-leaders/offense/passing.json
Season 2022: https://www.maxpreps.com/_next/data/1770310387/football/22-23/stat-leaders/offense/passing.json
Season 2023: https://www.maxpreps.com/_next/data/1770310387/football/23-24/stat-leaders/offense/passing.json
Season 2024: https: